# ContactEase — Rubrica di contatti

**ContactEase Solutions** — applicazione interattiva da terminale per gestire una
rubrica di contatti telefonici.

| # | Funzionalità |
|---|---|
| 1 | Aggiungere un contatto |
| 2 | Visualizzare tutti i contatti |
| 3 | Cercare un contatto per nome o cognome |
| 4 | Modificare un contatto esistente |
| 5 | Eliminare un contatto |
| 6 | Salvare i contatti su file (formato **JSON**) |
| 7 | Caricare automaticamente i contatti all'avvio |

## Come è organizzato il notebook

1. Introduzione (questa cella)
2. Installazione libreria e import
3. Classe `Contatto`
5. Classe `Rubrica` (dati, ricerca, persistenza)
9. Funzioni dell'interfaccia (libreria `rich`)
11. Funzione `main()` con il menu
13. Avvio dell'applicazione
14. Test automatici


## Path google frive e nome file memoria contatti

```python
from google.colab import drive
drive.mount('/content/drive')
NOME_FILE = '/content/drive/MyDrive/contatti.json'
```

## Import e installazione

Installiamo la libreria `rich` (serve per un'interfaccia a riga di comando
più leggibile: pannelli, tabelle, colori) e importiamo tutto ciò che serve.
Su Google Colab `rich` è quasi sempre già installata: il comando non farà
nulla in quel caso.

In [1]:
%pip install rich

import json
import os
import time
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich import box
from IPython.display import clear_output


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: C:\Users\carmi\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


## La classe `Contatto`

Rappresenta **un singolo contatto** della rubrica. È una classe semplice: tiene
insieme i dati di una persona e sa convertirsi da/verso un dizionario, il
formato usato per salvare su file JSON.

Attributi:

- `id` — numero identificativo, assegnato dalla `Rubrica` e **stabile** nel tempo
- `nome`, `cognome`, `numero` — dati obbligatori
- `email`, `indirizzo` — dati facoltativi (possono essere stringhe vuote)

Metodi:

- `to_dict()` — restituisce un dizionario con i campi del contatto
- `from_dict(d)` — ricostruisce un `Contatto` da un dizionario

In [2]:
class Contatto:
    """Rappresenta un singolo contatto della rubrica."""

    def __init__(self, id, nome, cognome, numero, email, indirizzo):
        self.id = id
        self.nome = nome
        self.cognome = cognome
        self.numero = numero
        self.email = email
        self.indirizzo = indirizzo

    def to_dict(self):
        """Converte il contatto in un dizionario pronto per il JSON."""
        return {
            "id": self.id,
            "nome": self.nome,
            "cognome": self.cognome,
            "numero": self.numero,
            "email": self.email,
            "indirizzo": self.indirizzo,
        }

    @staticmethod
    def from_dict(d):
        return Contatto(d["id"], d["nome"], d["cognome"],
                         d["numero"], d["email"], d["indirizzo"])

## La classe `Rubrica`

Conserva i contatti e si occupa di tutte
le operazioni, incluso il salvataggio su file.

Attributi:
- `percorso_file` — nome del file JSON dei contatti
- `contatti` — **dizionario**: chiave = `id` del contatto, valore = oggetto
  `Contatto`
- `prossimo_id` — prossimo `id` libero da assegnare. **Non viene salvato nel
  file**: viene ricalcolato ogni volta che la `Rubrica` viene creata, a
  partire dai contatti presenti (0 se è vuota, altrimenti `id massimo + 1`)

Metodi:

| Metodo | Cosa fa |
|---|---|
| `aggiungi(...)` | crea un nuovo contatto con l'`id` corrente, lo inserisce nel dizionario e aggiorna il contatore |
| `trova_per_id(id)` | restituisce il contatto con quell'`id`, oppure `None` |
| `modifica(id, ...)` | aggiorna i campi di un contatto; `True` se esiste |
| `elimina(id)` | rimuove un contatto; `True` se esisteva |
| `cerca(testo)` | contatti il cui **nome o cognome** contiene `testo` (maiuscole/minuscole indifferenti) |
| `elenco()` | tutti i contatti ordinati per cognome, poi nome |
| `is_vuota()` | `True` se non ci sono contatti |
| `salva()` | scrive i contatti nel file JSON |
| `carica()` | legge i contatti dal file JSON, se esiste ed è valido (chiamato automaticamente dal costruttore) |

### Formato del file `contatti.json`

```json
{
  "contatti": [
    {"id": 0, "nome": "Mario", "cognome": "Rossi",
     "numero": "+39 333 1234567", "email": "mario@example.com",
     "indirizzo": "Via Roma 1, Milano"}
  ]
}
```


In [3]:
class Rubrica:
    """Gestisce i contatti (in un dizionario id -> Contatto) e la loro persistenza."""

    def __init__(self, percorso_file=''):
        """Crea la rubrica.

        Se 'percorso_file' e' vuoto (il default), la rubrica parte vuota e
        non tocca il disco: comodo per crearne una "al volo", ad esempio nei
        test. Se invece viene passato un percorso, il costruttore chiama
        subito carica() per leggere i contatti gia' salvati in quel file.
        """
        self.percorso_file = percorso_file
        self.contatti = {}      # chiave: id (int) -> valore: oggetto Contatto
        self.prossimo_id = 0    # rubrica vuota: il primo id assegnato sarà 0
        if percorso_file:
            self.carica()

    def aggiungi(self, nome, cognome, numero, email, indirizzo):
        """Crea un nuovo contatto, lo inserisce nel dizionario e lo restituisce."""
        contatto = Contatto(self.prossimo_id, nome, cognome, numero, email, indirizzo)
        self.contatti[contatto.id] = contatto
        self.prossimo_id += 1
        return contatto

    def trova_per_id(self, id):
        """Restituisce il contatto con quell'id, oppure None (accesso diretto)."""
        return self.contatti.get(id)

    def modifica(self, id, nome, cognome, numero, email, indirizzo):
        """Aggiorna i campi del contatto con quell'id. True se esiste."""
        contatto = self.trova_per_id(id)
        if contatto is None:
            return False
        contatto.nome = nome
        contatto.cognome = cognome
        contatto.numero = numero
        contatto.email = email
        contatto.indirizzo = indirizzo
        return True

    def elimina(self, id):
        """Rimuove il contatto con quell'id. True se esisteva."""
        if id not in self.contatti:
            return False
        del self.contatti[id]
        return True

    def cerca(self, testo):
        """Contatti il cui nome O cognome contiene 'testo' (case-insensitive)."""
        testo = testo.strip().lower()
        if testo == "":
            return []
        return [c for c in self.contatti.values()
                if testo in c.nome.lower() or testo in c.cognome.lower()]

    def elenco(self):
        """Tutti i contatti ordinati per cognome, poi nome."""
        return sorted(self.contatti.values(),
                      key=lambda c: (c.cognome.lower(), c.nome.lower()))

    def is_vuota(self):
        """True se non ci sono contatti."""
        return len(self.contatti) == 0

    def salva(self):
        """Scrive i contatti su file JSON (UTF-8, leggibile).

        Non salviamo 'prossimo_id': viene ricalcolato ogni volta che la
        Rubrica viene creata (vedi carica()), a partire dagli id presenti.
        """
        dati = {
            "contatti": [c.to_dict() for c in self.contatti.values()],
        }
        with open(self.percorso_file, "w", encoding="utf-8") as f:
            json.dump(dati, f, indent=2, ensure_ascii=False)

    def carica(self):
        """Carica i contatti dal file JSON, se esiste ed e' valido.

        'prossimo_id' non viene letto dal file: e' sempre ricalcolato da qui,
        come 'id massimo tra i contatti presenti' + 1 (0 se la rubrica e'
        vuota). Significa che se elimini il contatto con l'id piu' alto e poi
        salvi e ricarichi, quell'id puo' essere riassegnato a un contatto
        nuovo: e' una scelta di semplicita', non un problema di correttezza.
        """
        if not os.path.exists(self.percorso_file):
            print(f"File '{self.percorso_file}' non trovato: rubrica vuota.")
            return
        try:
            with open(self.percorso_file, encoding="utf-8") as f:
                dati = json.load(f)
        except (json.JSONDecodeError, OSError):
            print(f"File '{self.percorso_file}' illeggibile o corrotto: "
                  f"rubrica vuota (il file non e' stato modificato).")
            return
        contatti_letti = [Contatto.from_dict(d) for d in dati.get("contatti", [])]
        self.contatti = {c.id: c for c in contatti_letti}
        if self.contatti:
            self.prossimo_id = max(self.contatti.keys()) + 1
        else:
            self.prossimo_id = 0

## Le funzioni dell'interfaccia

Sono funzioni separate dalle classi: si occupano **solo** di mostrare cose a
schermo e di leggere l'input dell'utente, usando la libreria `rich` per una
resa ordinata (pannelli, tabelle, colori).

### Principio: una schermata per volta

L'applicazione gira dentro l'output di **una sola cella** (quella con
`main()`, più avanti). Ogni volta che l'utente fa una scelta, quell'output
viene **ripulito** prima di mostrare il passo successivo. A schermo restano
solo:

1. l'**intestazione** dell'azione corrente;
2. le **informazioni necessarie in quel momento** — senza residui dei passi
   precedenti.

`intestazione(titolo)` fa proprio questo: pulisce lo schermo e stampa il
pannello del titolo.

### Come vengono lette le risposte

Tutte le domande passano da `chiedi_riga(messaggio)`, che stampa la domanda
nell'**output della cella** (così è visibile, non solo nella casella di
input del frontend) e poi legge una riga da tastiera. (**NB** : Il `flush` e
la pausa prima di `input()` servono a far si che la prima `input()` dopo un `clear_output()` restituisca subito una
stringa vuota, senza aspettare che l'utente digiti.)

### Validazione del campo "Numero"

`_chiedi_campo` accetta anche una funzione opzionale `valida`: se la
risposta digitata non la supera, la domanda viene ripetuta mostrando un
messaggio d'errore, invece di accettare qualunque testo. La usiamo solo per
il numero di telefono, con `_numero_valido(testo)`: accetta solo cifre, con
un eventuale `+` iniziale (es. `+39333123456` o `333123456`; rifiuta cose
come `33a` o un `+` da solo).

| Funzione | Ruolo |
|---|---|
| `pulisci_schermo()` | svuota l'output della cella |
| `intestazione(titolo)` | pulisce lo schermo e stampa il pannello del titolo |
| `chiedi_riga(messaggio)` | stampa la domanda e legge una riga da tastiera |
| `mostra_menu()` | elenca le voci del menu |
| `mostra_tabella(contatti, titolo)` | mostra i contatti in tabella |
| `_numero_valido(testo)` | `True` se `testo` è fatto solo di cifre, con un eventuale `+` iniziale |
| `chiedi_dati_contatto(correnti=None)` | chiede i 5 campi di un contatto |
| `leggi_intero(messaggio)` | legge un numero, ripetendo se l'input non è valido |
| `messaggio_ok / messaggio_errore / messaggio_info` | righe colorate di esito |
| `pausa()` | attende la pressione di Invio |

In [4]:
# Un unico oggetto Console condiviso da tutte le funzioni di stampa.
console = Console()

# Voci del menu principale.
VOCI_MENU = [
    "Aggiungi contatto",
    "Visualizza tutti i contatti",
    "Cerca contatto (per nome o cognome)",
    "Modifica contatto",
    "Elimina contatto",
    "Salva su file",
    "Esci",
]


def pulisci_schermo():
    """Svuota l'output della cella."""
    clear_output(wait=True)


def intestazione(titolo):
    """Pulisce lo schermo e stampa il pannello dell'azione corrente."""
    pulisci_schermo()
    console.print(Panel(f"[bold]{titolo}[/bold]",
                        title="[bold cyan]ContactEase[/bold cyan]",
                        border_style="cyan", box=box.DOUBLE))


def chiedi_riga(messaggio=""):
    """Stampa 'messaggio' nell'output della cella e legge una riga da tastiera."""
    print(messaggio+'\n', end="", flush=True)
    time.sleep(0.1)
    return input()


def mostra_menu():
    """Elenca le voci numerate del menu principale."""
    for numero, voce in enumerate(VOCI_MENU, start=1):
        console.print(f"  [bold yellow]{numero}[/bold yellow]) {voce}")
    console.print()


def mostra_tabella(contatti, titolo):
    """Mostra i contatti in una tabella; se vuota, un messaggio informativo."""
    if not contatti:
        messaggio_info("Nessun contatto da mostrare.")
        return
    tabella = Table(title=titolo, box=box.SIMPLE_HEAVY, title_style="bold")
    for colonna in ("ID", "Nome", "Cognome", "Numero", "Email", "Indirizzo"):
        tabella.add_column(colonna)
    for c in contatti:
        tabella.add_row(str(c.id), c.nome, c.cognome, c.numero, c.email, c.indirizzo)
    console.print(tabella)


def _numero_valido(testo):
    """True se 'testo' e' fatto solo di cifre, con un eventuale '+' iniziale."""
    corpo = testo[1:] if testo.startswith("+") else testo
    return corpo.isdigit()


def _chiedi_campo(etichetta, obbligatorio, valore_corrente=None, valida=None, errore_valida="Valore non valido."):
    """Chiede un singolo campo.

    - Invio (risposta vuota) mantiene 'valore_corrente' se fornito (modifica).
    - Se il campo e' facoltativo e la risposta e' vuota, restituisce "".
    - Se e' obbligatorio e non c'e' un valore corrente, ripete la domanda.
    - Se viene passata una funzione 'valida', la risposta digitata (non
      quella mantenuta con Invio) deve superarla, altrimenti si ripete la
      domanda mostrando 'errore_valida'.
    """
    suffisso = f" [{valore_corrente}]" if valore_corrente not in (None, "") else ""
    while True:
        risposta = chiedi_riga(f"{etichetta}{suffisso}: ").strip()
        if risposta == "" and valore_corrente is not None:
            return valore_corrente
        if risposta == "" and not obbligatorio:
            return ""
        if risposta != "":
            if valida is not None and not valida(risposta):
                console.print(f"  [red]{errore_valida}[/red]")
                continue
            return risposta
        console.print("  [red]Campo obbligatorio.[/red]")


def chiedi_dati_contatto(correnti=None):
    """Raccoglie i 5 campi di un contatto e li restituisce in un dizionario."""
    c = correnti or {}
    return {
        "nome":      _chiedi_campo("Nome", True, c.get("nome")),
        "cognome":   _chiedi_campo("Cognome", True, c.get("cognome")),
        "numero":    _chiedi_campo("Numero", True, c.get("numero"), valida=_numero_valido,
                                    errore_valida="Numero non valido: solo cifre, con un eventuale '+' iniziale."),
        "email":     _chiedi_campo("Email", False, c.get("email")),
        "indirizzo": _chiedi_campo("Indirizzo", False, c.get("indirizzo")),
    }


def leggi_intero(messaggio):
    """Legge un numero intero, ripetendo la domanda finche' non e' valido."""
    while True:
        risposta = chiedi_riga(f"{messaggio}: ").strip()
        if risposta == "":
            continue
        try:
            return int(risposta)
        except ValueError:
            console.print("  [red]Inserisci un numero intero.[/red]")


def messaggio_ok(testo):
    console.print(f"[bold green]OK[/bold green] {testo}")


def messaggio_errore(testo):
    console.print(f"[bold red]Errore[/bold red] {testo}")


def messaggio_info(testo):
    console.print(f"[cyan]{testo}[/cyan]")


def pausa():
    chiedi_riga("\nPremi Invio per continuare...")

## La funzione `main()`

Mette insieme tutti i pezzi:

1. crea una `Rubrica` passando `NOME_FILE`: il costruttore, ricevendo un
   percorso non vuoto, chiama da solo `carica()` per leggere i contatti
   salvati in precedenza;
2. entra nel **ciclo del menu**: mostra le opzioni, legge la scelta, esegue
   l'azione, attende Invio e ricomincia.

Ogni azione inizia con `intestazione(...)` (che pulisce lo schermo) e la
richiama prima di ogni passo successivo, così a video resta solo ciò che
serve in quel momento.

Il salvataggio è **esplicito**: la voce *6* salva su richiesta e, all'uscita
(voce *7*), viene chiesto se salvare le modifiche.

In [5]:
# Nome del file in cui vengono salvati i contatti.
# Su Colab, per una persistenza permanente, si può puntare a Google Drive
# (vedi la nota nella prima cella).
NOME_FILE = "contatti.json"


def main():
    """Avvia la rubrica (il costruttore carica già i dati da NOME_FILE) ed esegue il ciclo del menu."""
    rubrica = Rubrica(NOME_FILE)

    while True:
        intestazione("Menu principale")
        mostra_menu()
        scelta = leggi_intero("Seleziona un'opzione")

        if scelta == 1:
            intestazione("Aggiungi contatto")
            dati = chiedi_dati_contatto()
            contatto = rubrica.aggiungi(nome=dati["nome"], cognome=dati["cognome"],
                                        numero=dati["numero"], email=dati["email"],
                                        indirizzo=dati["indirizzo"])
            intestazione("Aggiungi contatto")
            messaggio_ok(f"Contatto aggiunto con ID {contatto.id}.")

        elif scelta == 2:
            intestazione("Tutti i contatti")
            mostra_tabella(rubrica.elenco(), "Tutti i contatti")

        elif scelta == 3:
            intestazione("Cerca contatto")
            testo = chiedi_riga("Nome o cognome da cercare: ")
            intestazione("Cerca contatto")
            mostra_tabella(rubrica.cerca(testo), f"Risultati per '{testo.strip()}'")

        elif scelta == 4:
            intestazione("Modifica contatto")
            mostra_tabella(rubrica.elenco(), "Tutti i contatti")
            id_scelto = leggi_intero("ID del contatto da modificare")
            contatto = rubrica.trova_per_id(id_scelto)
            if contatto is None:
                intestazione("Modifica contatto")
                messaggio_errore(f"Nessun contatto con ID {id_scelto}.")
            else:
                intestazione("Modifica contatto")
                console.print("Premi Invio per mantenere il valore attuale.\n")
                dati = chiedi_dati_contatto(correnti=contatto.to_dict())
                rubrica.modifica(id_scelto, **dati)
                intestazione("Modifica contatto")
                messaggio_ok(f"Contatto {id_scelto} aggiornato.")

        elif scelta == 5:
            intestazione("Elimina contatto")
            mostra_tabella(rubrica.elenco(), "Tutti i contatti")
            id_scelto = leggi_intero("ID del contatto da eliminare")
            intestazione("Elimina contatto")
            if rubrica.elimina(id_scelto):
                messaggio_ok(f"Contatto {id_scelto} eliminato.")
            else:
                messaggio_errore(f"Nessun contatto con ID {id_scelto}.")

        elif scelta == 6:
            intestazione("Salva su file")
            rubrica.salva()
            messaggio_ok(f"Contatti salvati in '{rubrica.percorso_file}'.")

        elif scelta == 7:
            intestazione("Esci")
            risposta = chiedi_riga("Salvare le modifiche? [s/n] (default s): ").strip().lower()
            if risposta in ("", "s", "si", "sì"):
                rubrica.salva()
                messaggio_ok("Modifiche salvate.")
            messaggio_info("Arrivederci!")
            break

        else:
            intestazione("Menu principale")
            messaggio_errore("Opzione non valida: scegli un numero da 1 a 7.")

        if scelta != 7:
            pausa()

# Esegui main

In [ ]:
# Avvia l'applicazione. Esegui questa cella e segui il menu.
# Per fermarla: scegli l'opzione 7 
main()

# Verifica automatica

Questa cella controlla che i metodi della classe `Rubrica` funzionino come
previsto. Usa un file temporaneo: **non tocca** `contatti.json`.

In [7]:
import tempfile

def _verifica():
    # 1) aggiungi() assegna id progressivi a partire da 0 e aggiorna il contatore
    #    Rubrica() senza argomenti crea una rubrica vuota, senza toccare il disco.
    r = Rubrica()
    r.aggiungi("Mario", "Rossi", "333", "", "")
    r.aggiungi("Anna", "Bianchi", "347", "", "")
    r.aggiungi("marco", "Rossini", "348", "", "")
    assert list(r.contatti.keys()) == [0, 1, 2]
    assert r.prossimo_id == 3

    # 1bis) una rubrica vuota parte da prossimo_id 0
    assert Rubrica().prossimo_id == 0
    assert Rubrica().is_vuota() is True

    # 2) cerca() per nome o cognome, senza distinzione maiuscole/minuscole
    assert {c.id for c in r.cerca("mar")} == {0, 2}
    assert {c.id for c in r.cerca("ROSS")} == {0, 2}
    assert r.cerca("   ") == []            # testo vuoto -> nessun risultato

    # 3) modifica() aggiorna un contatto esistente, altrimenti False
    assert r.modifica(1, "Anna", "Verdi", "999", "a@x.it", "Via A") is True
    assert r.trova_per_id(1).cognome == "Verdi"
    assert r.modifica(99, "x", "x", "x", "x", "x") is False

    # 4) elimina() rimuove un contatto esistente, altrimenti False
    assert r.elimina(0) is True
    assert r.trova_per_id(0) is None
    assert r.elimina(0) is False

    # 5) elenco() ordina per cognome, poi nome
    #    Restano: id 1 "Anna Verdi" e id 2 "marco Rossini" -> ordinati per cognome
    assert [c.cognome for c in r.elenco()] == ["Rossini", "Verdi"]

    # 6) salva() + carica(): i dati sopravvivono a un giro su file, e
    #    prossimo_id viene ricalcolato correttamente (non e' nel file).
    #    Passando il percorso al costruttore, Rubrica(percorso) carica da sola.
    percorso = tempfile.mktemp(suffix=".json")
    try:
        r.percorso_file = percorso
        r.salva()
        r2 = Rubrica(percorso)
        assert [c.to_dict() for c in r2.elenco()] == [c.to_dict() for c in r.elenco()]
        assert r2.prossimo_id == r.prossimo_id
    finally:
        if os.path.exists(percorso):
            os.remove(percorso)

    # 7) _numero_valido(): solo cifre, con un eventuale '+' iniziale
    assert _numero_valido("333") is True
    assert _numero_valido("+39333123456") is True
    assert _numero_valido("+") is False        # solo il '+', nessuna cifra
    assert _numero_valido("33a3") is False     # contiene una lettera
    assert _numero_valido("") is False         # stringa vuota

    print("✅ Tutti i test superati.")

_verifica()

✅ Tutti i test superati.
